## 1 · Setup
Clones the repository, checks `ffmpeg` and installs the dependencies (NeMo is skipped on purpose — it conflicts with `qwen-asr`).

| Option | Values |
|---|---|
| `UPDATE_REPO` | `True` pull the latest code on re-run · `False` keep local edits |

In [ ]:
%pip install -q -U "yt-dlp[default]"
%pip install -q qwen-asr librosa soundfile jiwer faster-whisper tqdm ipywidgets

# Deno = JavaScript runtime that recent yt-dlp needs to solve YouTube's challenges
if shutil.which("deno") is None:
    subprocess.run("curl -fsSL https://deno.land/install.sh | sh", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ["PATH"] = f"{os.path.expanduser('~/.deno/bin')}:{os.environ['PATH']}"
print("deno:", shutil.which("deno") or "NOT FOUND", "| yt-dlp:", shutil.which("yt-dlp"))

In [1]:
import gc
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ.setdefault("USE_TF", "0")

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
REPO_URL = "https://github.com/CaptainLevi04/Audio-Datasets-Creation-System.git"
UPDATE_REPO = True

if (WORK_ROOT / "speech_corpus").is_dir():
    REPO_DIR = WORK_ROOT
elif (Path.cwd() / "speech_corpus").is_dir():
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = WORK_ROOT / "Audio-Datasets-Creation-System"
    if not REPO_DIR.exists():
        print(f"📥 Cloning repository into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif UPDATE_REPO and (REPO_DIR / ".git").exists():
        print(f"🔄 Updating repository in {REPO_DIR} ...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(WORK_ROOT)
if str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

if shutil.which("ffmpeg") is None:
    print("ffmpeg not found on PATH; attempting installation ...")
    if sys.platform.startswith("linux"):
        subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg", shell=True, check=False)
    if shutil.which("ffmpeg") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"])


📥 Cloning repository into /kaggle/working/Audio-Datasets-Creation-System ...


Cloning into '/kaggle/working/Audio-Datasets-Creation-System'...


Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
deno: /root/.deno/bin/deno | yt-dlp: /usr/local/bin/yt-dlp


### Imports & helpers
Loads the `speech_corpus` modules, detects the compute device and defines two small shared helpers.

In [2]:
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from speech_corpus import ast_model, segmentation
from speech_corpus.ast_model import classify_audio, save_regions, validate_cached_regions
from speech_corpus.audio import DEFAULT_SR, read_wav, write_wav
from speech_corpus.cut_engine_v2 import cut_speech_and_mixed_segments
from speech_corpus.downloader import (
    download_video, extract_video_id, find_cached_video, ingest_local_file,
)
from speech_corpus.quality_control import (
    generate_qc_markdown_report, split_corpus, validate_segments_integrity, video_summary,
)
from speech_corpus.segmentation import detect_speech_activity_overlay
from speech_corpus.snr import compute_snr_for_segments, estimate_segment_snr, plot_snr_histogram
from speech_corpus.visualization import plot_segments_overview, plot_spectrogram_regions
from sautlab.kaggle_bridge import build_corpus_archive, ensure_conditioned_flac
from sautlab.remote_server import start_remote_server

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

device = "cuda" if torch.cuda.is_available() else "cpu"
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"


def banner(title, char="="):
    print(char * 72)
    print(title)
    print(char * 72)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"✅ Environment ready | ON_KAGGLE={ON_KAGGLE} | WORK_ROOT={WORK_ROOT}")
print(f"   Repo: {REPO_DIR} | Device: {device.upper()} ({dev_name})")

✅ Environment ready | ON_KAGGLE=True | WORK_ROOT=/kaggle/working
   Repo: /kaggle/working/Audio-Datasets-Creation-System | Device: CUDA (Tesla T4)


## 2 · Configuration
The only cell you need to edit — pass one or more videos (YouTube URLs or local files) and tune the options below.

| Option | Possible values |
|---|---|
| `SOURCES` | `"https://youtube.com/watch?v=..."` · `{"url": "...", "video_id": "lecture1"}` · `{"file": "/kaggle/input/<dataset>/lecture.mp4", "video_id": "lecture1"}` |
| `USE_MOCK_DEMO` | `True` runs an offline synthetic smoke test and ignores `SOURCES` |
| `COOKIES_FILE` | `None` · path to a YouTube `cookies.txt` |
| `ATTACHED_OUTPUT_ROOT` | `None` · `Path("/kaggle/input/<dataset>/corpus_data")` to reuse cached outputs |
| `FORCE_RECLASSIFY` / `FORCE_RECUT` / `FORCE_RESNR` / `FORCE_RETRANSCRIBE` | `True` ignores the cache and recomputes that stage |
| `ACCEPT_QC_FAILED_SEGMENTS` | `True` continues when Cut Engine QC fails (segments are promoted from `segments_staging/`) · `False` stops with an error |
| `USE_WORD_VERIFICATION` | `True` enables the faster-whisper word-boundary check during cutting (`WORD_MODEL_SIZE`: `"small"`, `"medium"`, …) |
| `SNR_THRESHOLD_DB` | clean/noisy cut-off in dB — `25.0` default, the spec starts at `20.0` |
| `SNR_COMBINE_MODE` | `"mean"` · `"wada"` · `"energy"` · `"min"` |
| `ASR_LIMIT` | `None` for all `speech`/`mixed` segments · e.g. `10` for a quick test |
| `WHISPER_BATCH` / `AUDAR_BATCH` | lower to `2`–`1` / `4` on CUDA out-of-memory |
| `ASR_ACCEPT_CER` / `ASR_HUMAN_MAX_WORD_DIFF` | agreement rule: CER ≤ first → `accepted`; word diffs ≤ second → `human_choice`; otherwise `difficult` |
| `SPLIT_RATIOS` | `(train, val, test)` — splits are created only with ≥ 3 videos |
| `ZIP_MODE` | `"compact"` (lossless FLAC + CSVs, ~4–5x smaller, auto-hydrated locally) · `"full"` (WAVs) · `"metadata"` (CSVs only) |
| `START_REMOTE_SERVER` | `True` exposes a Kaggle GPU HTTP worker for local `run_gui.py --kaggle-url <url>` · `False` skips |
| `REMOTE_PORT` / `REMOTE_TUNNEL` | Port (`8765`) and tunnel (`"cloudflared"` or `None`) when `START_REMOTE_SERVER = True` |

In [3]:
SOURCES = [
    {"url": "https://www.youtube.com/watch?v=9BdEKLR0kUE"},
]

USE_MOCK_DEMO = False
# Cookies: put your cookies.txt in a Kaggle dataset. It is auto-detected under /kaggle/input.
# Set USE_COOKIES = False to disable.
USE_COOKIES = True
COOKIES_FILE = None
if USE_COOKIES:
    import glob
    _found = glob.glob("/kaggle/input/**/*cookies*.txt", recursive=True)
    if _found:
        # /kaggle/input is read-only and yt-dlp writes back to the cookie file -> use a writable copy
        COOKIES_FILE = Path("/kaggle/working/.cookies.txt")
        shutil.copy(_found[0], COOKIES_FILE)
        print(f"🍪 Using cookies from {_found[0]} (writable copy: {COOKIES_FILE})")
    else:
        print("⚠️  No *cookies*.txt found under /kaggle/input — downloads may hit YouTube's bot check.")
ATTACHED_OUTPUT_ROOT = None

DATA_ROOT = WORK_ROOT / "corpus_data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
SR = DEFAULT_SR

SILENCE_THRESHOLD_S = 0.25
FORCE_RECLASSIFY = False
FORCE_RECUT = False
ACCEPT_QC_FAILED_SEGMENTS = True
USE_WORD_VERIFICATION = False
WORD_MODEL_SIZE = "small"

SNR_THRESHOLD_DB = 25.0
SNR_COMBINE_MODE = "mean"
FORCE_RESNR = False

ASR_LABELS = ("speech", "mixed")
ASR_LIMIT = None
FORCE_RETRANSCRIBE = False

WHISPER_ID = "openai/whisper-large-v3"
WHISPER_LANG = "ar"
WHISPER_BATCH = 4
WHISPER_BEAMS = 5

AUDAR_ID = "audarai/Audar-ASR-V1-Turbo"
AUDAR_LANG = "Arabic"
AUDAR_BATCH = 8

ASR_ACCEPT_CER = 0.02
ASR_HUMAN_MAX_WORD_DIFF = 3
SPLIT_RATIOS = (0.70, 0.15, 0.15)

ZIP_MODE = "compact"
START_REMOTE_SERVER = False
REMOTE_PORT = 8765
REMOTE_TUNNEL = "cloudflared"

if ATTACHED_OUTPUT_ROOT is not None:
    src_root = Path(ATTACHED_OUTPUT_ROOT)
    if src_root.exists():
        for item in src_root.iterdir():
            dst = DATA_ROOT / item.name
            if item.is_dir() and not dst.exists():
                shutil.copytree(item, dst)
        print(f"📂 Copied cached outputs from {src_root} -> {DATA_ROOT.resolve()}")

print(f"⚙️  Configured {len(SOURCES)} source(s) | USE_MOCK_DEMO={USE_MOCK_DEMO} | DATA_ROOT={DATA_ROOT.resolve()}")

🍪 Using cookies from /kaggle/input/datasets/amrgamal007/yt-cookies/www.youtube.com_cookies.txt (writable copy: /kaggle/working/.cookies.txt)
⚙️  Configured 1 source(s) | USE_MOCK_DEMO=False | DATA_ROOT=/kaggle/working/corpus_data


## 3 · Phase 1 — Download, Label & Segment
Defines the source resolver, audio ingestion and the QC-recovery step that promotes `segments_staging/` to `segments/`.

In [4]:
def resolve_local_file(file_str):
    p = Path(file_str)
    if p.exists():
        return p
    if Path("/kaggle/input").exists():
        matches = list(Path("/kaggle/input").rglob(p.name))
        if matches:
            print(f"Resolved local file from Kaggle input: {matches[0]}")
            return matches[0]
    raise FileNotFoundError(f"Could not find media file: {file_str}")


def prepare_jobs():
    if USE_MOCK_DEMO:
        return [{"mode": "mock", "video_id": "mock_demo_audio"}]
    jobs = []
    for idx, entry in enumerate(SOURCES, 1):
        if isinstance(entry, str):
            entry = {"url": entry}
        if entry.get("file"):
            fp = resolve_local_file(entry["file"])
            vid = entry.get("video_id") or extract_video_id(str(fp))
            jobs.append({"mode": "file", "file": fp, "video_id": vid})
        elif entry.get("url"):
            vid = entry.get("video_id") or extract_video_id(entry["url"])
            jobs.append({"mode": "url", "url": entry["url"], "video_id": vid})
        else:
            raise ValueError(f"Invalid SOURCES entry #{idx}: {entry}")
    return jobs


def download_video_cli(url, out_dir, cookies_file=None):
    """Download audio via the yt-dlp CLI (always the freshly installed version) -> audio.wav (16 kHz mono) + info.json."""
    import time
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_path, info_path = out_dir / "audio.wav", out_dir / "info.json"

    cmd = ["yt-dlp", "--no-playlist", "-f", "bestaudio/best", "--write-info-json",
           "-o", str(out_dir / "temp_audio.%(ext)s"), url]
    if cookies_file:
        cmd[1:1] = ["--cookies", str(cookies_file)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"yt-dlp failed:\n{res.stderr}")

    raw = out_dir / "temp_audio.info.json"
    meta = json.loads(raw.read_text(encoding="utf-8")) if raw.exists() else {}
    temp_files = [f for f in out_dir.glob("temp_audio.*") if not f.name.endswith(".json")]
    if not temp_files:
        raise RuntimeError("Downloaded audio file not found.")

    subprocess.run(["ffmpeg", "-y", "-i", str(temp_files[0]), "-ar", "16000", "-ac", "1", str(wav_path)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    for f in out_dir.glob("temp_audio.*"):
        f.unlink(missing_ok=True)

    lic = str(meta.get("license") or "").lower()
    info = {
        "video_id": out_dir.name, "source": "youtube", "url": url,
        "webpage_url": meta.get("webpage_url") or url,
        "title": meta.get("title") or "Unknown Title",
        "channel": meta.get("channel") or meta.get("uploader") or "Unknown Channel",
        "duration_s": meta.get("duration"), "upload_date": meta.get("upload_date"),
        "license": meta.get("license") or "Standard YouTube License",
        "is_creative_commons": ("creative commons" in lic) or ("cc-by" in lic) or ("public domain" in lic),
        "downloaded_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    info_path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    return wav_path, info


def ingest_audio(job, video_dir):
    audio_path = video_dir / "audio.wav"
    info_path = video_dir / "info.json"

    if audio_path.exists() and info_path.exists():
        print(f"Found cached audio.wav & info.json in {video_dir}")
        info = json.loads(info_path.read_text(encoding="utf-8"))
    elif job["mode"] == "mock":
        from speech_corpus.labelling import build_mock_audio
        samples, _ = build_mock_audio(seed=42, sr=SR)
        write_wav(audio_path, samples, sample_rate=SR)
        info = {
            "video_id": video_dir.name, "source": "mock", "url": None,
            "title": "Synthetic Audio Demo", "channel": "Synthesizer",
            "duration_s": round(len(samples) / SR, 2),
            "license": "Creative Commons Attribution", "is_creative_commons": True,
        }
        info_path.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
        return samples, info, video_dir
    elif job["mode"] == "file":
        audio_path, info = ingest_local_file(job["file"], video_dir)
    else:
        cached = find_cached_video(job["url"], search_dirs=[DATA_ROOT])
        if cached is not None:
            audio_path, info = cached
            video_dir = audio_path.parent
        else:
            audio_path, info = download_video_cli(job["url"], video_dir, cookies_file=COOKIES_FILE)

    samples, _ = read_wav(audio_path, target_sr=SR)
    return samples, info, video_dir


def finalize_segments(segments_df, video_dir):
    segments_dir = video_dir / "segments"
    staging_dir = video_dir / "segments_staging"

    if not segments_dir.exists() and staging_dir.exists():
        if not ACCEPT_QC_FAILED_SEGMENTS:
            raise RuntimeError(
                f"Cut Engine v2 QC failed for {video_dir.name}; segments were left in {staging_dir}. "
                "Fix the QC errors or set ACCEPT_QC_FAILED_SEGMENTS = True."
            )
        staging_dir.rename(segments_dir)
        for col in ("audio_file", "wav_file"):
            if col in segments_df.columns:
                segments_df[col] = segments_df[col].map(
                    lambda x: x.replace("segments_staging/", "segments/") if isinstance(x, str) else x
                )
        print(f"⚠️  QC failed for {video_dir.name}: promoted segments_staging/ -> segments/ "
              f"(see {video_dir / 'qc_report.md'} for the failed gates)")
    elif not segments_dir.exists():
        raise RuntimeError(f"Cut Engine v2 produced no segments for {video_dir.name}")

    segments_df.to_csv(video_dir / "segments_student1.csv", index=False)
    return segments_df

Defines the per-video pipeline: ingest → 16 kHz WAV → VAD pauses → AST labels → Cut Engine v2 → 10-gate QC.

In [5]:
def run_student1(job, idx, total):
    vid = job["video_id"]
    video_dir = DATA_ROOT / vid
    video_dir.mkdir(parents=True, exist_ok=True)
    banner(f"[Phase 1 | Video {idx}/{total}] video_id={vid} -> {video_dir}")

    audio, info, video_dir = ingest_audio(job, video_dir)
    dur_s = len(audio) / SR
    print(f"Audio ready: {dur_s:.1f}s ({dur_s / 60:.2f} min) @ {SR} Hz | License: {info.get('license')}")

    backup_csv = video_dir / "segments_student1.csv"
    regions_csv = video_dir / "regions.csv"
    windows_csv = video_dir / "windows_ast.csv"

    _, pauses_df = detect_speech_activity_overlay(
        audio=audio, sr=SR,
        silence_threshold_s=SILENCE_THRESHOLD_S,
        min_pause_ms=120, vad_backend="auto",
    )

    if (not FORCE_RECLASSIFY) and validate_cached_regions(
        regions_csv, expected_source_id=video_dir.name, expected_duration=dur_s, expected_version="2.2"
    ):
        print(f"Loaded cached regions (v2.2): {regions_csv}")
        regions_df = pd.read_csv(regions_csv)
    else:
        print("Running AST macro-region classification ...")
        regions_df, windows_df = classify_audio(
            y=audio, sr=SR, win_s=10.0, hop_s=5.0, batch_size=32, pauses_df=pauses_df
        )
        save_regions(regions_df, regions_csv, source_id=video_dir.name, duration=dur_s, version="2.2")
        if windows_df is not None and not windows_df.empty:
            windows_df.to_csv(windows_csv, index=False)

    spec_path = video_dir / "full_spectrogram.png"
    if not spec_path.exists() or FORCE_RECLASSIFY:
        fig = plot_spectrogram_regions(
            audio, regions_df, sr=SR,
            title=f"Full Audio Spectrogram - {str(info.get('title', vid))[:40]}",
            save_path=spec_path,
        )
        plt.close(fig)

    if (not FORCE_RECUT) and backup_csv.exists() and (video_dir / "segments").exists():
        segments_df = pd.read_csv(backup_csv, dtype={"segment_id": str})
        print(f"Loaded cached segments ({len(segments_df)} rows): {backup_csv}")
    else:
        segments_df = cut_speech_and_mixed_segments(
            audio=audio,
            regions_df=regions_df,
            out_dir=video_dir,
            sr=SR,
            pauses=pauses_df,
            use_word_verification=USE_WORD_VERIFICATION,
            word_model_size=WORD_MODEL_SIZE,
            run_ablation=True,
        )
        segments_df = finalize_segments(segments_df, video_dir)

    overview_path = video_dir / "segments_overview.png"
    if not overview_path.exists() or FORCE_RECUT:
        fig = plot_segments_overview(audio, segments_df, sr=SR, max_segments=20, save_path=overview_path)
        plt.close(fig)

    qc = validate_segments_integrity(segments_df, video_dir=video_dir, total_dur=dur_s, sr=SR)
    generate_qc_markdown_report(segments_df, report_path=video_dir / "qc_report.md", video_dir=video_dir, qc=qc)
    vsum_df = video_summary(regions_df=regions_df, segments_df=segments_df, info=info, video_dir=video_dir)

    lbl_counts = segments_df["label"].astype(str).str.lower().value_counts().to_dict()
    print(f"QC status: {qc['status']} ({qc['summary']['passed_gates']}/{qc['summary']['total_gates']} gates)")
    print(f"Segments: {len(segments_df)} total | By label: {lbl_counts}")
    display(vsum_df)

    del audio
    gc.collect()
    return video_dir

Runs Phase 1 for every source and saves `audio.wav`, `regions.csv`, `segments/*.wav`, `qc_report.md` and `segments_student1.csv`.

In [6]:
jobs = prepare_jobs()
video_dirs = [run_student1(job, i, len(jobs)) for i, job in enumerate(jobs, 1)]

ast_model._AST_CACHE.clear()
segmentation._SILERO.clear()
free_gpu()
print(f"\n✅ Phase 1 complete. Outputs saved under: {DATA_ROOT.resolve()}")

[Phase 1 | Video 1/1] video_id=9BdEKLR0kUE -> /kaggle/working/corpus_data/9BdEKLR0kUE
Audio ready: 533.4s (8.89 min) @ 16000 Hz | License: Standard YouTube License
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
Running AST macro-region classification ...
[AST] Loading 'MIT/ast-finetuned-audioset-10-10-0.4593' on CUDA...


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[AST] class ids -> speech:[0, 1, 2, 3, 4, 5] music:[27, 137, 138, 266] silence:[500]
ℹ️ Loaded regions.csv (includes macro-regions and any human edits).
🎯 CUT ENGINE V2 — UNIFIED FOUR-LABEL PIPELINE [PASSED]
• Total segments generated     : 40 (total audio: 8.89 min)
• Primary window (5–15 s)      : 40 / 40 (100.0%)
• Fallback window (3–18 s)     : 0 / 40
• Margin 100–200 ms satisfied  : 100.0%
• Whisper word checks          : verified 0, split risks 0
• Review workload              : 0 / 40 (0.0%) [target 5–15%]
• Mandatory QC gates passed    : 10 / 10
• Output manifest              : /kaggle/working/corpus_data/9BdEKLR0kUE/segments.csv
Saved visual report image: /kaggle/working/corpus_data/9BdEKLR0kUE/segments_overview.png
QC status: PASSED (10/10 gates)
Segments: 40 total | By label: {'speech': 40}


,video,duration_min,speech_min,music_min,mixed_min,silence_min,segments,mean_seg_s,review_pct,license,url
0,اجتماع واشنطن وخيارات التصعيد مع طهران - نقاش ...,8.89,8.89,0.0,0.0,0.0,40,13.33,0.0,Standard YouTube License,https://www.youtube.com/watch?v=9BdEKLR0kUE



✅ Phase 1 complete. Outputs saved under: /kaggle/working/corpus_data


## 4 · Phase 2 — SNR Estimation
Computes Energy and WADA SNR for **all** segments and saves `segments_snr.csv`, `snr_diagnostics.csv` and both histograms.

| Option | Possible values |
|---|---|
| `SNR_THRESHOLD_DB` | clean/noisy cut-off in dB (default `25.0`) |
| `SNR_COMBINE_MODE` | `"mean"` · `"wada"` · `"energy"` · `"min"` |
| `FORCE_RESNR` | `True` recomputes even when cached |

In [7]:
def save_length_histogram(durations, path, title):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(durations.dropna(), bins=25, color="#2ca02c", edgecolor="black", alpha=0.85)
    ax.axvline(5.0, color="red", ls="--", label="5s min")
    ax.axvline(15.0, color="red", ls="--", label="15s max")
    ax.set_xlabel("Segment duration (s)")
    ax.set_ylabel("Number of segments")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


snr_dirs = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and (p / "segments").is_dir()
    and ((p / "segments_student1.csv").exists() or (p / "segments.csv").exists())
)
assert snr_dirs, f"No segmented videos found in {DATA_ROOT.resolve()}. Run Phase 1 first."

for video_dir in snr_dirs:
    banner(f"[Phase 2 | SNR] {video_dir}")

    s1_csv = video_dir / "segments_student1.csv"
    src_csv = s1_csv if s1_csv.exists() else (video_dir / "segments.csv")
    seg_src_df = pd.read_csv(src_csv, dtype={"segment_id": str})

    if "wav_file" not in seg_src_df.columns and "audio_file" in seg_src_df.columns:
        seg_src_df["wav_file"] = seg_src_df["audio_file"].map(
            lambda x: f"segments/{Path(str(x)).name}" if pd.notnull(x) and not (video_dir / str(x)).exists() else x
        )

    lbl_counts = seg_src_df["label"].astype(str).str.lower().value_counts().to_dict()
    print(f"Passing ALL {len(seg_src_df)} segments to SNR: {lbl_counts}")

    snr_ckpt = video_dir / "segments_snr.csv"
    diag_csv = video_dir / "snr_diagnostics.csv"

    if (not FORCE_RESNR) and snr_ckpt.exists() and diag_csv.exists():
        snr_df = pd.read_csv(snr_ckpt, dtype={"segment_id": str})
        print(f"Loaded cached SNR results: {snr_ckpt}")
    else:
        snr_df, _ = compute_snr_for_segments(
            seg_src_df,
            base_dir=video_dir,
            threshold_db=SNR_THRESHOLD_DB,
            combine=SNR_COMBINE_MODE,
            save_csv=True,
            verbose=True,
        )
        snr_df.to_csv(snr_ckpt, index=False, encoding="utf-8-sig")

        diag_rows = []
        for _, row in seg_src_df.iterrows():
            rel_wav = row["wav_file"] if "wav_file" in seg_src_df.columns else row["audio_file"]
            wav_path = video_dir / Path(str(rel_wav))
            if not wav_path.exists():
                wav_path = video_dir / "segments" / Path(str(rel_wav)).name
            samples, sr_w = read_wav(wav_path, target_sr=SR)
            est = estimate_segment_snr(samples, sr=sr_w, threshold_db=SNR_THRESHOLD_DB, combine=SNR_COMBINE_MODE)
            diag_rows.append({
                "segment_id": row["segment_id"],
                "label": row["label"],
                "snr_energy_db": est["snr_energy_db"],
                "snr_wada_db": est["snr_wada_db"],
                "snr": est["snr"],
                "quality": est["quality"],
                "snr_disagree": est["snr_disagree"],
            })
        pd.DataFrame(diag_rows).to_csv(diag_csv, index=False, encoding="utf-8-sig")

    fig = plot_snr_histogram(snr_df, threshold_db=SNR_THRESHOLD_DB, save_path=video_dir / "snr_histogram.png")
    plt.close(fig)

    durations = pd.to_numeric(snr_df["end"], errors="coerce") - pd.to_numeric(snr_df["start"], errors="coerce")
    save_length_histogram(
        durations, video_dir / "segment_length_histogram.png",
        f"Segment-Length Distribution ({video_dir.name})",
    )

    mean_snr = pd.to_numeric(snr_df["snr"], errors="coerce").mean()
    print(f"Quality counts: {snr_df['quality'].value_counts().to_dict()} | Mean SNR: {mean_snr:.2f} dB")
    display(snr_df[["segment_id", "start", "end", "label", "snr", "quality"]].head(8))

print("\n✅ Phase 2 complete.")

[Phase 2 | SNR] /kaggle/working/corpus_data/9BdEKLR0kUE
Passing ALL 40 segments to SNR: {'speech': 40}
Step 5: 40 segments | clean=19 noisy=21 | threshold=25.0 dB | mean SNR=26.2 dB | methods disagree on 30
Quality counts: {'noisy': 21, 'clean': 19} | Mean SNR: 26.20 dB


,segment_id,start,end,label,snr,quality
0,1,0.00,13.70,speech,22.92,noisy
1,2,13.70,23.75,speech,26.15,clean
2,3,23.75,35.45,speech,28.54,clean
3,4,35.45,47.85,speech,24.66,noisy
4,5,47.85,62.55,speech,24.82,noisy
5,6,62.55,74.95,speech,23.04,noisy
6,7,74.95,85.75,speech,24.53,noisy
7,8,85.75,100.10,speech,22.38,noisy



✅ Phase 2 complete.


## 5 · Phase 3 — Dual ASR
Keeps only `speech`/`mixed` segments (music and silence skip ASR) and loads their audio into memory.

| Option | Possible values |
|---|---|
| `ASR_LABELS` | labels sent to ASR — default `("speech", "mixed")` |
| `ASR_LIMIT` | `None` for all segments · e.g. `10` for a quick test |

In [8]:
def resolve_segment_wav(video_dir, raw_path):
    p = Path(str(raw_path))
    candidates = [
        p if p.is_absolute() else None,
        WORK_ROOT / p,
        video_dir / p,
        video_dir / "segments" / p.name,
        video_dir / "segments_staging" / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return c
    raise FileNotFoundError(f"Missing segment WAV '{raw_path}' for {video_dir}")


def load_asr_checkpoint(vdir, df_asr, ckpt_name, column):
    ckpt = vdir / ckpt_name
    if FORCE_RETRANSCRIBE or not ckpt.exists():
        return False
    c_df = pd.read_csv(ckpt, dtype={"segment_id": str})
    same_rows = len(c_df) == len(df_asr) and [str(x) for x in c_df["segment_id"]] == [str(x) for x in df_asr["segment_id"]]
    if not same_rows:
        return False
    df_asr[column] = c_df[column].fillna("").tolist()
    return True


asr_dirs = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and (p / "segments_snr.csv").exists())
assert asr_dirs, f"No segments_snr.csv found under {DATA_ROOT.resolve()}. Run Phase 2 first."

allowed_labels = {s.lower() for s in ASR_LABELS}
asr_jobs = {}

for video_dir in asr_dirs:
    snr_df = pd.read_csv(video_dir / "segments_snr.csv", dtype={"segment_id": str})
    snr_df["_row_idx"] = np.arange(len(snr_df))
    lbl_norm = snr_df["label"].astype(str).str.strip().str.lower()

    asr_mask = lbl_norm.isin(allowed_labels)
    df_asr_all = snr_df[asr_mask].copy().reset_index(drop=True)
    df_non_asr = snr_df[~asr_mask].copy().reset_index(drop=True)

    if ASR_LIMIT is not None:
        df_asr = df_asr_all.head(ASR_LIMIT).copy().reset_index(drop=True)
        df_asr_skipped = df_asr_all.iloc[ASR_LIMIT:].copy().reset_index(drop=True)
    else:
        df_asr = df_asr_all
        df_asr_skipped = pd.DataFrame(columns=snr_df.columns)

    if not df_asr.empty:
        bad = set(df_asr["label"].astype(str).str.strip().str.lower()) - allowed_labels
        assert not bad, f"Unexpected labels passed to ASR: {bad}"

    print(
        f"[{video_dir.name}] Total={len(snr_df)} | ASR={len(df_asr)} | Non-ASR={len(df_non_asr)}"
        + (f" | Deferred(ASR_LIMIT)={len(df_asr_skipped)}" if len(df_asr_skipped) else "")
    )

    audio_list = []
    for row in df_asr.itertuples():
        x, _ = read_wav(resolve_segment_wav(video_dir, row.audio_file), target_sr=SR)
        audio_list.append(x)

    durs = np.array([len(a) / SR for a in audio_list], dtype=float)
    if len(durs):
        assert durs.max() <= 30.5, (
            f"[{video_dir.name}] Segment exceeds 30.5s (max={durs.max():.2f}s); ASR models support <= 30s."
        )
        print(f"  Loaded {len(audio_list)} clips ({durs.sum() / 60:.1f} min, max={durs.max():.1f}s)")
    rms = np.array([float(np.sqrt((a ** 2).mean())) for a in audio_list], dtype=float)

    asr_jobs[video_dir] = {
        "df_asr": df_asr,
        "df_non_asr": df_non_asr,
        "df_asr_skipped": df_asr_skipped,
        "audio": audio_list,
        "rms": rms,
    }

[9BdEKLR0kUE] Total=40 | ASR=40 | Non-ASR=0
  Loaded 40 clips (8.9 min, max=15.0s)


**Model A — Whisper large-v3:** transcribes the clips, saves `asr_whisper_ckpt.csv` per video and frees the GPU.

| Option | Possible values |
|---|---|
| `WHISPER_BATCH` | `4` default — lower to `2` or `1` on CUDA out-of-memory |
| `WHISPER_BEAMS` / `WHISPER_LANG` | beam-search width · language code (`"ar"`) |
| `FORCE_RETRANSCRIBE` | `True` ignores existing checkpoints |

In [9]:
need_whisper = []
for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"]
    if df_asr.empty:
        df_asr["model_A_transcript"] = []
    elif load_asr_checkpoint(vdir, df_asr, "asr_whisper_ckpt.csv", "model_A_transcript"):
        print(f"[{vdir.name}] Loaded cached Whisper checkpoint")
    else:
        need_whisper.append(vdir)

if need_whisper:
    from transformers import AutoProcessor, WhisperForConditionalGeneration

    w_dev = "cuda" if torch.cuda.is_available() else "cpu"
    w_dtype = torch.float16 if w_dev == "cuda" else torch.float32
    print(f"Loading {WHISPER_ID} on {w_dev} (dtype={w_dtype}) ...")
    w_proc = AutoProcessor.from_pretrained(WHISPER_ID)
    w_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, torch_dtype=w_dtype).to(w_dev).eval()

    for vdir in need_whisper:
        job = asr_jobs[vdir]
        audio_list, rms = job["audio"], job["rms"]
        whisper_txt = []
        print(f"Transcribing {vdir.name} ({len(audio_list)} segments) with Whisper large-v3 ...")
        for i in tqdm(range(0, len(audio_list), WHISPER_BATCH)):
            batch = audio_list[i:i + WHISPER_BATCH]
            feats = w_proc(batch, sampling_rate=SR, return_tensors="pt").input_features.to(w_dev, dtype=w_dtype)
            with torch.inference_mode():
                ids = w_model.generate(
                    feats, language=WHISPER_LANG, task="transcribe",
                    num_beams=WHISPER_BEAMS, max_new_tokens=300,
                )
            texts = w_proc.batch_decode(ids, skip_special_tokens=True)
            for k, t in enumerate(texts):
                whisper_txt.append("" if rms[i + k] < 1e-3 else t.strip())

        job["df_asr"]["model_A_transcript"] = whisper_txt
        job["df_asr"][["_row_idx", "segment_id", "label", "model_A_transcript"]].to_csv(
            vdir / "asr_whisper_ckpt.csv", index=False, encoding="utf-8-sig"
        )
        print(f"Saved Whisper checkpoint: {vdir / 'asr_whisper_ckpt.csv'}")

    del w_model, w_proc
    free_gpu()
    if torch.cuda.is_available():
        print(f"Freed GPU after Whisper | Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Whisper skipped (everything cached or no speech/mixed segments).")

Loading openai/whisper-large-v3 on cuda (dtype=torch.float16) ...


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (7 outputs left out)

Transcribing 9BdEKLR0kUE (40 segments) with Whisper large-v3 ...


  0%|          | 0/10 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Saved Whisper checkpoint: /kaggle/working/corpus_data/9BdEKLR0kUE/asr_whisper_ckpt.csv
Freed GPU after Whisper | Allocated: 0.01 GB


**Model B — Audar-ASR-V1-Turbo:** transcribes the same clips, saves `asr_audar_ckpt.csv` per video and frees the GPU.

| Option | Possible values |
|---|---|
| `AUDAR_BATCH` | `8` default — lower to `4` on CUDA out-of-memory |
| `AUDAR_LANG` | language name passed to the model (`"Arabic"`) |

In [10]:
need_audar = []
for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"]
    if df_asr.empty:
        df_asr["model_B_transcript"] = []
    elif load_asr_checkpoint(vdir, df_asr, "asr_audar_ckpt.csv", "model_B_transcript"):
        df_asr["second_model"] = "audar"
        print(f"[{vdir.name}] Loaded cached Audar checkpoint")
    else:
        need_audar.append(vdir)

if need_audar:
    from qwen_asr import Qwen3ASRModel

    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        a_dtype = torch.bfloat16 if major >= 8 else torch.float16
        a_device = "cuda:0"
    else:
        a_dtype = torch.float32
        a_device = "cpu"

    print(f"Loading {AUDAR_ID} on {a_device} (dtype={a_dtype}) ...")
    a_model = Qwen3ASRModel.from_pretrained(
        AUDAR_ID, dtype=a_dtype, device_map=a_device,
        max_inference_batch_size=AUDAR_BATCH, max_new_tokens=256,
    )
    a_model.model.to(a_dtype)

    for vdir in need_audar:
        job = asr_jobs[vdir]
        audio_list, rms = job["audio"], job["rms"]
        audar_txt = []
        print(f"Transcribing {vdir.name} ({len(audio_list)} segments) with Audar-ASR-V1-Turbo ...")
        for i in tqdm(range(0, len(audio_list), AUDAR_BATCH)):
            batch = [(a, SR) for a in audio_list[i:i + AUDAR_BATCH]]
            out = a_model.transcribe(audio=batch, language=[AUDAR_LANG] * len(batch))
            audar_txt.extend(r.text.strip() for r in out)

        job["df_asr"]["second_model"] = "audar"
        job["df_asr"]["model_B_transcript"] = [("" if r < 1e-3 else t) for t, r in zip(audar_txt, rms)]
        job["df_asr"][["_row_idx", "segment_id", "label", "model_B_transcript"]].to_csv(
            vdir / "asr_audar_ckpt.csv", index=False, encoding="utf-8-sig"
        )
        print(f"Saved Audar checkpoint: {vdir / 'asr_audar_ckpt.csv'}")

    del a_model
    free_gpu()
    if torch.cuda.is_available():
        print(f"Freed GPU after Audar | Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Audar skipped (everything cached or no speech/mixed segments).")

Loading audarai/Audar-ASR-V1-Turbo on cuda:0 (dtype=torch.float16) ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

chat_template.json: 0.00B [00:00, ?B/s]

Transcribing 9BdEKLR0kUE (40 segments) with Audar-ASR-V1-Turbo ...


  0%|          | 0/5 [00:00<?, ?it/s]

Saved Audar checkpoint: /kaggle/working/corpus_data/9BdEKLR0kUE/asr_audar_ckpt.csv
Freed GPU after Audar | Allocated: 0.01 GB


Defines Arabic text normalisation and the agreement rule between the two transcripts.

| Option | Possible values |
|---|---|
| `ASR_ACCEPT_CER` | CER at or below this → `accepted` (default `0.02`) |
| `ASR_HUMAN_MAX_WORD_DIFF` | word differences at or below this → `human_choice`, otherwise `difficult` (default `3`) |

In [11]:
DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ALEF_MAP = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"})

APP_COLS = [
    "start", "end", "label", "snr", "quality",
    "model_A_transcript", "model_B_transcript", "final_transcript",
    "status", "review_status", "automatic_status",
    "segment_id", "audio_file",
    "changed_by", "changed_at", "change_log",
]

BLANK_META = {
    "model_A_transcript": "", "model_B_transcript": "", "final_transcript": "",
    "cer": np.nan, "n_diff_words": np.nan, "second_model": "",
    "changed_by": "", "changed_at": "", "change_log": "[]",
}


def normalize_ar(t):
    t = "" if t is None or (isinstance(t, float) and t != t) else str(t)
    t = DIACRITICS.sub("", t).replace("ـ", "").translate(ALEF_MAP)
    t = t.replace("ة", "ه").replace("ى", "ي")
    t = re.sub(r"[^\w\s]", " ", t).replace("_", " ")
    return " ".join(t.split())


def compare_transcripts(a, b):
    na, nb = normalize_ar(a), normalize_ar(b)
    if not na or not nb:
        return "difficult", 1.0, 999
    w = jiwer.process_words(na, nb)
    n_diff = int(w.substitutions + w.deletions + w.insertions)
    cer = float(jiwer.cer(na, nb))
    if cer <= ASR_ACCEPT_CER:
        return "accepted", round(cer, 4), n_diff
    status = "human_choice" if n_diff <= ASR_HUMAN_MAX_WORD_DIFF else "difficult"
    return status, round(cer, 4), n_diff


def score_asr_rows(df):
    results = [compare_transcripts(a, b) for a, b in zip(df["model_A_transcript"], df["model_B_transcript"])]
    statuses = [r[0] for r in results]
    df = df.copy()
    df["automatic_status"] = statuses
    df["status"] = statuses
    df["review_status"] = ["accepted" if s == "accepted" else "pending" for s in statuses]
    df["final_transcript"] = [a if s == "accepted" else "" for a, s in zip(df["model_A_transcript"], statuses)]
    df["cer"] = [r[1] for r in results]
    df["n_diff_words"] = [r[2] for r in results]
    df["second_model"] = "audar"
    df["changed_by"] = ""
    df["changed_at"] = ""
    df["change_log"] = "[]"
    return df


def blank_asr_rows(df, status, review_status):
    df = df.copy()
    for col, val in BLANK_META.items():
        df[col] = val
    df["status"] = status
    df["automatic_status"] = status
    df["review_status"] = review_status
    return df

Merges ASR rows with music/silence rows and writes `segments.csv` (16-column app format), `segments_full.csv`, `segments_asr.csv` and `transcripts/*.txt`.

In [12]:
for vdir, job in asr_jobs.items():
    df_asr = score_asr_rows(job["df_asr"]) if len(job["df_asr"]) else job["df_asr"]
    df_non_asr = blank_asr_rows(job["df_non_asr"], "no_asr", "skipped") if len(job["df_non_asr"]) else job["df_non_asr"]
    df_skipped = blank_asr_rows(job["df_asr_skipped"], "not_run", "pending") if len(job["df_asr_skipped"]) else job["df_asr_skipped"]

    parts = [p for p in (df_asr, df_non_asr, df_skipped) if len(p) > 0]
    merged_df = pd.concat(parts, ignore_index=True).sort_values("_row_idx").reset_index(drop=True)

    df_out = merged_df[[c for c in APP_COLS if c in merged_df.columns]].copy()
    df_out.to_csv(vdir / "segments.csv", index=False, encoding="utf-8-sig")
    df_out.to_csv(vdir / "segments_asr.csv", index=False, encoding="utf-8-sig")

    full_df = merged_df.copy()
    s1_path = vdir / "segments_student1.csv"
    diag_path = vdir / "snr_diagnostics.csv"
    if s1_path.exists():
        s1_df = pd.read_csv(s1_path, dtype={"segment_id": str}).reset_index(drop=True)
        if len(s1_df) == len(full_df):
            if "status" in s1_df.columns:
                full_df["qc_status"] = s1_df["status"].values
            for c in [c for c in s1_df.columns if c not in full_df.columns]:
                full_df[c] = s1_df[c].values
    if diag_path.exists():
        d_df = pd.read_csv(diag_path, dtype={"segment_id": str}).reset_index(drop=True)
        if len(d_df) == len(full_df):
            for c in ["snr_energy_db", "snr_wada_db", "snr_disagree"]:
                if c in d_df.columns:
                    full_df[c] = d_df[c].values
    full_df.drop(columns=["_row_idx"], errors="ignore").to_csv(
        vdir / "segments_full.csv", index=False, encoding="utf-8-sig"
    )

    txt_dir = vdir / "transcripts"
    txt_dir.mkdir(parents=True, exist_ok=True)
    for r in df_asr.itertuples():
        txt_path = txt_dir / f"segment_{r.segment_id}.txt"
        if r.status == "accepted":
            txt_path.write_text(str(r.final_transcript), encoding="utf-8")
        else:
            txt_path.write_text(
                f"[{r.status}]\nModel A (Whisper): {r.model_A_transcript}\nModel B (Audar): {r.model_B_transcript}\n",
                encoding="utf-8",
            )

    ensure_conditioned_flac(vdir)
    print(f"\n[{vdir.name}] Status breakdown: {df_out['status'].value_counts().to_dict()}")
    print(f"  Saved: {vdir / 'segments.csv'} ({len(df_out.columns)} cols)")
    print(f"  Saved: {vdir / 'segments_full.csv'} ({len(full_df.columns) - 1} cols)")
    display(df_out[["segment_id", "label", "snr", "quality", "model_A_transcript", "model_B_transcript", "status"]].head(10))

print("\n✅ Phase 3 complete.")


[9BdEKLR0kUE] Status breakdown: {'accepted': 19, 'human_choice': 12, 'difficult': 9}
  Saved: /kaggle/working/corpus_data/9BdEKLR0kUE/segments.csv (16 cols)
  Saved: /kaggle/working/corpus_data/9BdEKLR0kUE/segments_full.csv (56 cols)


,segment_id,label,snr,quality,model_A_transcript,model_B_transcript,status
0,1,speech,22.92,noisy,من المهم التذكير بأن هذا الاجتماع لا يعقد بشكل...,أولا من المهم التذكير بأنه هذا الاجتماع لا يعق...,human_choice
1,2,speech,26.15,clean,وهي الفترة التي سبقت الحرب الإسرائيلية على إير...,وهي الفترة التي سبقت الحرب الإسرائيلية على إير...,human_choice
2,3,speech,28.54,clean,هذا واحد ثنين أنه الاجتماع ضم جميع التشكيلات ي...,هذا واحد، اثنين أنه الاجتماع ضم جميع التشكيلات...,accepted
3,4,speech,24.66,noisy,وأيضاً الاستخبارات مدير الـ CIA راتليف أيضاً ك...,وأيضا الاستخبارات مدير السي أي إيه راتلر كليف ...,difficult
4,5,speech,24.82,noisy,هذا يعني أن كل الاتجاهات موجودة في داخل الإدار...,هذا يعني أن كل الاتجاهات موجودة في داخل الإدار...,human_choice
5,6,speech,23.04,noisy,لأنه لو كان قرار بالتصعيد العسكري كان هناك حضو...,لأنه لو كان قرار بالتصعيد العسكري كان هناك حضو...,human_choice
6,7,speech,24.53,noisy,إقدام أو قبول لمرحلة جديدة وأنه نحن في مرحلة ا...,إقدام أو قبول لمرحلة جديدة وأنه نحن في مرحلة ا...,accepted
7,8,speech,22.38,noisy,أستبعد أن يكون هناك قرار صدر بهذا الاجتماع على...,أستبعد أن يكون هناك قرار صدر بهذا الاجتماع على...,accepted
8,9,speech,26.58,clean,بالنسبة إلى الرئيس ترامب في حالة التصعيد التي ...,بالنسبة إلى الرئيس ترمب ، في حالة التصعيد التي...,human_choice
9,10,speech,24.16,noisy,مع المملكة العربية السعودية باعتبارها الحليف و...,مع المملكة العربية السعودية باعتبارها الحليف و...,human_choice



✅ Phase 3 complete.


## 6 · Phase 4 — Local Desktop GUI (Student 3 · SautLab)
All heavy processing (download with Kaggle `/kaggle/input` cookies + `deno`, audio extraction, AST, Cut Engine v2, SNR, and Dual ASR) stays on Kaggle, while **SautLab** (`run_gui.py`) runs natively on your PC with its full desktop UI and minimal data transfer.

- **Batch mode (`START_REMOTE_SERVER = False`):** Run Phase 5 below to download the compact archive (`ZIP_MODE = "compact"`, ~4–5× smaller lossless FLAC + CSVs) and open it locally:
  `python run_gui.py --import-zip corpus_final_output.zip`
- **Interactive Remote GPU mode (`START_REMOTE_SERVER = True`):** Starts a Kaggle GPU server using your `/kaggle/input` cookies automatically so you can paste new YouTube URLs or run Steps 1–6 directly from `python run_gui.py --kaggle-url <url>`.

In [13]:
if START_REMOTE_SERVER:
    start_remote_server(
        data_dir=DATA_ROOT,
        port=REMOTE_PORT,
        tunnel=REMOTE_TUNNEL,
        cookies_source=COOKIES_FILE,
        work_root=WORK_ROOT,
        block=True,
    )
else:
    print("⏭️  START_REMOTE_SERVER is False — skipping remote GPU server.")
    print("   Run Phase 5 below, download corpus_final_output.zip, then run locally:")
    print("   python run_gui.py --import-zip corpus_final_output.zip")

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1201: UserWarning: Expected 6 arguments for function <function build_gradio_app.<locals>.on_table_select at 0x7834618b5260>, received 5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1205: UserWarning: Expected at least 6 arguments for function <function build_gradio_app.<locals>.on_table_select at 0x7834618b5260>, received 5.
  warnings.warn(


Launching SautLab Web GUI for data_dir=/kaggle/working/corpus_data (annotator='Team')...
* Running on public URL: https://adb913682530eb3dd6.gradio.live


Public Kaggle GUI URL: https://adb913682530eb3dd6.gradio.live
Local URL: http://127.0.0.1:7860/
Click 'Finish Review & Continue' inside the GUI (or press Kaggle Stop) when done reviewing.


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


GUI wait interrupted by user; saving any unsaved changes...
Closing server running on port: 7860


## 7 · Phase 5 — Final Packaging (Post-Processing / Post-Review)
Rebuilds the Step 7 summary table, global manifests, zero-leakage `train`/`val`/`test` splits, and `/kaggle/working/corpus_final_output.zip`.

| Option | Possible values |
|---|---|
| `SPLIT_RATIOS` | `(0.70, 0.15, 0.15)` default — splits are only created with ≥ 3 videos |
| `ZIP_MODE` | `"compact"` (lossless `audio_conditioned.flac` + CSVs, auto-hydrated by `run_gui.py`) · `"full"` · `"metadata"` |

In [14]:
video_folders = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and ((p / "segments_asr.csv").exists() or (p / "segments.csv").exists()) and (p / "regions.csv").exists()
)
assert video_folders, "No video folders found. Run Phases 1-3 first."

summary_rows, app_parts, full_parts = [], [], []

for vdir in video_folders:
    info_path = vdir / "info.json"
    info = json.loads(info_path.read_text(encoding="utf-8")) if info_path.exists() else {}
    reg_df = pd.read_csv(vdir / "regions.csv")
    seg_app = pd.read_csv(vdir / "segments.csv", dtype={"segment_id": str})
    app_parts.append(seg_app.assign(video_id=vdir.name))

    full_path = vdir / "segments_full.csv"
    if full_path.exists():
        full_df = pd.read_csv(full_path, dtype={"segment_id": str})
        if "video_id" not in full_df.columns:
            full_df.insert(0, "video_id", vdir.name)
        full_parts.append(full_df)

    reg_dur = reg_df.assign(d=reg_df["end"] - reg_df["start"]).groupby("label")["d"].sum() / 60.0
    asr_sub = seg_app[seg_app["label"].astype(str).str.lower().isin(["speech", "mixed"])].copy()
    n_asr = max(1, len(asr_sub))
    st_counts = asr_sub["status"].value_counts().to_dict() if len(asr_sub) else {}
    auto_st = asr_sub.get("automatic_status", pd.Series("", index=asr_sub.index)).astype(str)
    chg_by = asr_sub.get("changed_by", pd.Series("", index=asr_sub.index)).fillna("").astype(str).str.strip()
    chosen_hand_cnt = int(((asr_sub["status"] == "accepted") & ((auto_st != "accepted") | (chg_by != ""))).sum()) if len(asr_sub) else 0
    auto_acc_cnt = int(((asr_sub["status"] == "accepted") & (auto_st == "accepted") & (chg_by == "")).sum()) if len(asr_sub) else 0

    changes_csv = vdir / "changes.csv"
    n_overrides = int(len(pd.read_csv(changes_csv))) if changes_csv.exists() else 0
    mean_snr = pd.to_numeric(seg_app["snr"], errors="coerce").mean()

    summary_rows.append({
        "video_id": vdir.name,
        "title": info.get("title", vdir.name),
        "duration_min": round(float(reg_df["end"].max()) / 60.0, 2) if len(reg_df) else 0.0,
        "speech_min": round(float(reg_dur.get("speech", 0.0)), 2),
        "music_min": round(float(reg_dur.get("music", 0.0)), 2),
        "mixed_min": round(float(reg_dur.get("mixed", 0.0)), 2),
        "silence_min": round(float(reg_dur.get("silence", 0.0)), 2),
        "total_segments": len(seg_app),
        "asr_segments": len(asr_sub),
        "mean_snr_db": round(float(mean_snr), 2) if pd.notnull(mean_snr) else None,
        "clean_pct": round(100.0 * float((seg_app["quality"] == "clean").mean()), 1) if len(seg_app) else 0.0,
        "auto_accepted_pct": round(100.0 * auto_acc_cnt / n_asr, 1) if len(asr_sub) else 0.0,
        "chosen_by_hand_pct": round(100.0 * chosen_hand_cnt / n_asr, 1) if len(asr_sub) else 0.0,
        "accepted_total_pct": round(100.0 * st_counts.get("accepted", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "human_choice_pending_pct": round(100.0 * st_counts.get("human_choice", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "difficult_pct": round(100.0 * st_counts.get("difficult", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "human_overrides": n_overrides,
        "license": info.get("license"),
        "url": info.get("url"),
    })

step7_df = pd.DataFrame(summary_rows)
step7_df.to_csv(DATA_ROOT / "corpus_summary_step7.csv", index=False, encoding="utf-8-sig")
print("📊 Step 7 corpus summary (post-review):")
display(step7_df)

pd.concat(app_parts, ignore_index=True).to_csv(
    DATA_ROOT / "corpus_manifest_app.csv", index=False, encoding="utf-8-sig"
)

if full_parts:
    manifest_full = pd.concat(full_parts, ignore_index=True)
    if "duration" not in manifest_full.columns:
        manifest_full["duration"] = (
            pd.to_numeric(manifest_full["end"]) - pd.to_numeric(manifest_full["start"])
        ).round(3)
    manifest_full.to_csv(DATA_ROOT / "corpus_manifest_full.csv", index=False, encoding="utf-8-sig")
    splits = split_corpus(manifest_full, out_dir=DATA_ROOT / "splits", ratios=SPLIT_RATIOS)
    print("Split status:", splits.get("status"))

📊 Step 7 corpus summary (post-review):


,video_id,title,duration_min,speech_min,music_min,mixed_min,silence_min,total_segments,asr_segments,mean_snr_db,clean_pct,auto_accepted_pct,chosen_by_hand_pct,accepted_total_pct,human_choice_pending_pct,difficult_pct,human_overrides,license,url
0,9BdEKLR0kUE,اجتماع واشنطن وخيارات التصعيد مع طهران - نقاش ...,8.89,8.89,0.0,0.0,0.0,40,40,26.2,47.5,47.5,0.0,47.5,30.0,22.5,0,Standard YouTube License,https://www.youtube.com/watch?v=9BdEKLR0kUE


⚠️ LEAKAGE-SAFE SPLIT INFEASIBLE: Found 1 source(s), but at least 3 independent sources are required for a zero-leakage (train/val/test) split. Refusing to generate segment-level split to prevent data leakage.
Split status: INFEASIBLE


Packages the final corpus into `/kaggle/working/corpus_final_output.zip` using `ZIP_MODE` (`"compact"` cuts archive size ~4–5× while keeping lossless audio for local `run_gui.py`).

In [15]:
zip_path = WORK_ROOT / "corpus_final_output.zip"
build_corpus_archive(DATA_ROOT, zip_path, mode=ZIP_MODE)

print(f"📦 Final archive ready: {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MB)")
print(f"   Archive mode: {ZIP_MODE}")
print(f"   Open locally: python run_gui.py --import-zip {zip_path.name}")
print(f"   All files remain under: {DATA_ROOT.resolve()}")

📦 Final archive ready: /kaggle/working/corpus_final_output.zip (32.43 MB)
   Includes audio.wav: True
   All files remain under: /kaggle/working/corpus_data
